In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from numpy.linalg import eig
import pandas as pd

#import matplotlib  
#matplotlib.use('Agg')  # Use a non-GUI backend

In [2]:
cat='C6'
path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'
#path=f'C:/shehani/postdoc_work/ML_Diffusion/cal_md/new_cals/{cat}_rel/'

#df = pd.DataFrame(columns=['nrep', '#hops' 'Dnonhop_mean', 'Dnonhop_se', 'D_hop', 'Dtot'])

In [3]:
def oh_xyz(nrep, nsteps):
    #nrep = number of replicas
    #nsteps = number of steps in fs/MD_freq (ex: for 10 ps simulation with MD_freq=10, nsteps=10,000/10=1000)

    #extracting x,y, z cordinates from oh_id.dat files
    x_oh = np.zeros((nrep,nsteps))
    y_oh = np.zeros((nrep,nsteps))
    z_oh = np.zeros((nrep,nsteps))
    indx_oh= np.zeros((nrep,nsteps))

    for i in range(nrep):
        with open(path+ f'oh_id_{i+1}.dat', 'r') as oh_id:
            xyz= oh_id.readlines()[:nsteps]
            
            for j in range(len(xyz)):   
                indx_oh[i,j]=int(xyz[j].split()[1])
                x_oh[i,j]=float(xyz[j].split()[2])
                y_oh[i,j]=float(xyz[j].split()[3])
                z_oh[i,j]=float(xyz[j].split()[4])
                
    #dtime=  np.arange(0.01, (0.01*ndt+0.01), 0.01)
    return(x_oh, y_oh, z_oh, indx_oh)




In [4]:
nrep=15

nsteps=20000

x_oh, y_oh, z_oh, indx_oh= oh_xyz(nrep, nsteps)

In [5]:
dtime = np.zeros(nsteps)
D1x=np.zeros(nrep)
D1y=np.zeros(nrep)
D1=np.zeros(nrep)
rx=[]
ry=[]
t=[]
ax=[]
ay=[]
t_hop=[]
tau=[]
for ll in range(nrep):
    int_indx= indx_oh[ll,0]
    int_x= x_oh[ll,0]
    int_y= y_oh[ll,0]
    int_z= z_oh[ll,0]
    int_t=0
    hop=0
    for jj in range(nsteps):
        dtime[jj]= np.round(jj*0.01, 2)
        
        if indx_oh[ll,jj]==int_indx:
            pass
            #print('nonhop',ll, jj, indx_oh[ll,jj],dtime[jj], x_oh[ll,jj])
        else:
            hop=hop+1
            #t_hop.append(dtime[jj])
            rx.append((x_oh[ll,jj-1]-int_x)**2)
            ry.append((y_oh[ll,jj-1]-int_y)**2)
            t.append(dtime[jj-1]-int_t)
            #print('hop',ll, jj, indx_oh[ll,jj])
            #print(int_t, int_indx, int_x, int_y)
            #print(dtime[jj-1], indx_oh[ll,jj-1], x_oh[ll,jj-1], y_oh[ll,jj-1])     
            #print(t[-1], rx[-1], ry[-1])   
            ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2)
            ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2)
            print(jj)

            if hop>1:
                #ax.append((x_oh[ll,jj]-x_oh[ll,jj-1])**2)
                #ay.append((y_oh[ll,jj]-y_oh[ll,jj-1])**2)
                #print(ll,jj,(x_oh[ll,jj]-x_oh[ll,jj-1])**2 )
                #print(x_oh[ll,jj],x_oh[ll,jj-1])
                #print(dtime[jj-1],int_t)
                t_hop.append(dtime[jj-1]-int_t)

            
            int_indx= indx_oh[ll,jj]
            int_x= x_oh[ll,jj]
            int_y= y_oh[ll,jj]
            int_z= z_oh[ll,jj]
            int_t=dtime[jj]
            #print(hop,int_t)

            #print(indx_oh[ll,jj], x_oh[ll,jj], x_oh[ll,jj-1], dtime[jj],int_t)
            
    print(ll,hop)
    rx.append((x_oh[ll,-1]-int_x)**2)
    ry.append((y_oh[ll,-1]-int_y)**2)
    t.append(dtime[-1]-int_t)
    
    D1x[ll]=(np.mean(rx))/(2*np.mean(t))
    D1y[ll]=(np.mean(ry))/(2*np.mean(t))
    D1[ll]=(np.mean(rx)+np.mean(ry))/(4*np.mean(t))

dmean= np.mean(D1)
dstd= np.std(D1)
std_err= dstd/np.sqrt(nrep)

D2x=(np.mean(ax))/(2*np.mean(t_hop))
D2y=(np.mean(ay))/(2*np.mean(t_hop))
D2=(np.mean(ax)+np.mean(ay))/(4*np.mean(t_hop))
D=dmean+D2
                    
print('nohops:',dmean, std_err)
print('hops:',D2)
print('Total:',D)
print(D1)

220
595
1082
1638
2089
2335
2710
2775
3382
3742
3796
4211
4406
4929
5775
6780
7366
7687
7726
7764
8076
8818
10288
10392
13808
15594
16188
16665
16771
16796
18568
0 31
115
2073
2306
3808
3816
3842
4801
6482
7101
7300
7496
9639
10433
11709
11727
11747
13311
13493
14053
14304
14372
14919
15295
15737
16588
18174
1 26
1409
2601
2764
3460
3867
4291
5738
5825
5949
6436
6563
7048
7417
7688
7783
8042
8137
9506
9531
9564
9933
10274
11503
12019
12039
13210
13994
15628
15726
17008
17327
19421
2 32
1325
2750
4263
6027
6545
7429
11657
14695
16178
16488
17055
17961
18017
18076
19275
3 15
1165
1911
3955
4048
6528
7593
9046
9960
9995
16694
4 10
264
479
1373
1393
1699
2887
5082
5102
5259
9198
11588
13199
15444
16400
16703
17188
5 16
2301
5058
5229
6934
7646
13361
13540
13579
13613
13885
13956
14500
15705
6 13
59
6407
7336
10080
10725
13825
17856
18572
18611
7 9
2701
5452
6343
6493
6650
8535
8705
11984
12716
12793
16476
17462
18331
18596
19068
8 15
5279
5538
8850
10941
10967
16036
17077
18053
18350
19130

In [6]:
len(t_hop)+nrep

280

In [7]:
dmean= np.mean(D1)
dstd= np.std(D1)
std_err= dstd/np.sqrt(nrep)
dmean, dstd, std_err

(0.738363956584734, 0.05879962720052764, 0.015181998460723213)

In [8]:
c2=0.9627078511609803
c4= 0.8756260083700356
c6=0.738363956584734

In [9]:
np.mean(ax)

2.325785739388071

In [10]:
np.mean(ay)

2.2822045982830366

In [11]:
len(ax)

280

In [12]:
len(ay)

280

In [13]:
len(t_hop)

265

In [14]:
D2x

0.11915346652318735

In [15]:
(np.mean(ax))

2.325785739388071

In [16]:
np.mean(t_hop)

9.759622641509432

In [17]:
D2y

0.11692073977206913

In [18]:
D2

0.11803710314762825